# CSAKD YOLO26n — Standalone Kaggle / Colab

**Eksperimen 09** · Adaptasi pilot CSAKD: cross supervising + AAWM + AGKA pada fitur backbone/neck.

Notebook ini memuat **satu eksperimen**. Dataset awal `coco8.yaml` hanyalah uji integrasi (bukan hasil penelitian). Setelah dataset penelitian siap, ubah `DATA` dan parameter training di section 1. Audit dataset berada **di luar notebook ini**.

**Urutan:** jalankan notebook 01 dahulu; lalu gunakan `weights/best.pt` teacher yang sama untuk eksperimen 04–09. **Adaptasi CrossKD/CSAKD bukan reproduksi identik paper.**

**Demo W&B:** COCO8 (4 gambar train / 4 val), 10 epoch; model checkpoint setiap epoch sebagai W&B Artifact berversi. Angka mAP **bukan** hasil penelitian.


## 1. Experiment Configuration

In [ ]:
# Ubah parameter di bagian ini saja untuk menjalankan eksperimen.
EXPERIMENT_ID = "09_csakd_yolo26n"
METHOD = "csakd"
DATA = "/kaggle/input/poltrada-v3-yolo26-clean/data.yaml"  # Dataset bersih di Kaggle.
MODEL_WEIGHTS = "yolo26n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEED = 42
DEVICE = 0                   # GPU Kaggle; 'cpu' untuk diagnosis lokal
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
PATIENCE = 100
AMP = True

WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
EVAL_SPLIT = "val"           # test hanya jika dataset punya split test
PROJECT = "/kaggle/working/results"  # Colab: ganti ke /content/results
TEACHER_CHECKPOINT = ""   # WAJIB: best.pt hasil eksperimen 01 pada DATA yang sama
KD_AGKA_WEIGHT = 1.0
KD_CROSS_WEIGHT = 1.0

# W&B online: setiap epoch mengunggah last.pt + best.pt sampai epoch tersebut.
WANDB_ENABLED = True
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_ENTITY = None          # gunakan akun default; ganti nama tim bila perlu


## 2. Environment Setup

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys
import os

# Pertahankan instalasi PyTorch/CUDA bawaan Kaggle; instal hanya yang dibutuhkan.
for package, version in (("ultralytics", "8.4.155"), ("wandb", "0.28.1")):
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        print(f"Installing {package}=={version} (was {installed})")
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])

import json
import time
from datetime import datetime
from pathlib import Path

import torch
import wandb
from ultralytics import YOLO, settings
from ultralytics.utils.torch_utils import init_seeds

# Satu sumber logging W&B: callback eksplisit di section 6, bukan integrasi otomatis Ultralytics.
settings.update({"wandb": False})
init_seeds(SEED, deterministic=True)
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | Ultralytics {metadata.version('ultralytics')} | W&B {metadata.version('wandb')}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
if DEVICE != "cpu" and not torch.cuda.is_available():
    raise RuntimeError("GPU tidak tersedia. Aktifkan GPU Kaggle/Colab atau set DEVICE='cpu' untuk test lokal.")

if WANDB_ENABLED:
    # Kaggle: tambahkan secret WANDB_API_KEY melalui Add-ons > Secrets dan aktifkan notebook access.
    if not os.environ.get("WANDB_API_KEY"):
        try:
            from kaggle_secrets import UserSecretsClient
            secret = UserSecretsClient().get_secret("WANDB_API_KEY")
            if secret:
                os.environ["WANDB_API_KEY"] = secret
        except ImportError:
            pass  # Colab atau lokal: sediakan WANDB_API_KEY sebagai environment variable.
    if not os.environ.get("WANDB_API_KEY"):
        raise RuntimeError("W&B online membutuhkan WANDB_API_KEY. Set sebagai Kaggle Secret atau environment variable; jangan tulis token ke notebook.")
    wandb.login(key=os.environ["WANDB_API_KEY"])


## 3. Dataset Loading

In [ ]:
from ultralytics.data.utils import check_det_dataset

# Memakai loader resmi, tanpa audit/cleaning dataset dalam notebook eksperimen.
data_info = check_det_dataset(DATA)
assert int(data_info["nc"]) > 0, "Dataset tidak memiliki kelas deteksi."
assert data_info.get("train") and data_info.get("val"), "Dataset wajib punya train dan val."
if EVAL_SPLIT == "test" and not data_info.get("test"):
    raise ValueError("Dataset ini tidak memiliki test split; gunakan EVAL_SPLIT='val' hanya untuk diagnosis.")

# Periksa kesesuaian nama kelas teacher/dataset untuk metode KD.
def class_names(names):
    if isinstance(names, dict):
        return [str(names[k]) for k in sorted(names, key=lambda k: int(k))]
    return [str(x) for x in names]

print(f"Dataset: {DATA} | classes: {data_info['nc']} | validation: {EVAL_SPLIT}")

## 4. Model Initialization

In [ ]:
if not TEACHER_CHECKPOINT:
    raise ValueError("Isi TEACHER_CHECKPOINT dengan best.pt dari notebook 01 yang dilatih pada dataset yang sama.")
teacher_path = Path(TEACHER_CHECKPOINT).expanduser()
if not teacher_path.is_file():
    raise FileNotFoundError(f"Checkpoint teacher tidak ditemukan: {teacher_path}")
# Model teacher digunakan sebagai target; tidak dilatih ulang oleh notebook student.
teacher_meta = YOLO(str(teacher_path))
if int(teacher_meta.model.nc) != int(data_info["nc"]):
    raise ValueError("Jumlah kelas teacher dan dataset tidak sama.")
if class_names(teacher_meta.model.names) != class_names(data_info["names"]):
    raise ValueError("Urutan nama kelas teacher dan dataset berbeda.")
print("Teacher:", teacher_path, "| classes:", teacher_meta.model.nc)
del teacher_meta

model = YOLO(MODEL_WEIGHTS)
print("Student / model:", MODEL_WEIGHTS)

## 5. Knowledge Distillation Method

**Adaptasi eksperimen, bukan implementasi identik paper remote sensing.** Cross supervision memakai modul teacher yang dibekukan; AAWM/AGKA menggunakan attention pada fitur BCHW YOLO26 dengan pemetaan row-wise. Jika graph tidak cocok proses berhenti. Rujukan: *Knowledge Distillation via Cross Supervising with Attention for Remote Sensing Object Detection* (Zhan dkk.).

In [ ]:
import math
import torch
from torch import nn
from torch.nn import functional as F
from ultralytics.nn.modules.head import Detect
from ultralytics.nn.distill_model import DistillationModel, FeatureHook
from ultralytics.models.yolo.detect import DetectionTrainer


def detect_head(network):
    heads = [m for m in network.model if isinstance(m, Detect)]
    if len(heads) != 1:
        raise ValueError("Adaptasi KD ini mensyaratkan tepat satu Detect head.")
    return heads[0]


def align_channels(cin, cout):
    return nn.Identity() if cin == cout else nn.Conv2d(cin, cout, 1, bias=False)


def combined_loss(wrapper, batch, preds, weights):
    """Detection loss + KD loss; gradient KD hanya mengoptimasi student/adapter."""
    wrapper._teacher_feats.clear()
    wrapper._student_feats.clear()
    try:
        if not wrapper.training:
            base, items = wrapper.student_model.loss(batch, preds)
            kd_parts = {key: base.new_zeros(()) for key in weights}
        else:
            with torch.no_grad():
                wrapper.teacher_model(batch["img"])
            predictions = wrapper.student_model(batch["img"])
            base, items = wrapper.student_model.loss(batch, predictions)
            with torch.autocast(device_type=batch["img"].device.type, enabled=False):
                kd_parts = wrapper.kd_loss()
        kd = sum(weights[key] * kd_parts[key] for key in weights)
        if not torch.isfinite(kd).all():
            raise RuntimeError("KD loss non-finite; proses dihentikan.")
        items.update({key: value.detach() for key, value in kd_parts.items()})
        items["dis_loss"] = kd.detach()
        return torch.cat((base.reshape(-1), (kd * batch["img"].shape[0]).reshape(1))), items
    finally:
        wrapper._teacher_feats.clear()
        wrapper._student_feats.clear()

In [ ]:
def module_indices(network):
    # Tiga lateral backbone + tiga neck input; bergantung pada graph YOLO26.
    boundary = len(network.yaml["backbone"])
    lateral = set()
    for module in network.model[boundary:]:
        refs = module.f if isinstance(module.f, list) else [module.f]
        lateral.update(index for index in refs if 0 <= index < boundary)
    neck = list(detect_head(network).f)
    indices = sorted(lateral) + neck
    if len(lateral) != 3 or len(neck) != 3 or len(set(indices)) != 6:
        raise ValueError("Adaptasi CSAKD memerlukan 3 lateral backbone dan 3 output neck.")
    return indices


class InputHook:
    def __init__(self, cache, key):
        self.cache, self.key = cache, key

    def __call__(self, module, inputs):
        if len(inputs) != 1 or not isinstance(inputs[0], torch.Tensor):
            raise ValueError("Input modul CSAKD harus satu tensor.")
        self.cache[self.key] = inputs[0]


def csakd_losses(student, teacher, cross):
    """AAWM + AGKA; attention BCHW memakai adaptasi row-wise dari paper."""
    if student.shape != teacher.shape or teacher.shape != cross.shape:
        raise ValueError("Fitur CSAKD tidak seukuran setelah channel projection.")
    s, t, c = student.float(), teacher.detach().float(), cross.float()
    scale = math.sqrt(s.shape[1])
    similarity = (t.mean((2, 3)) * s.mean((2, 3))).sum(dim=1) / scale
    attention = torch.softmax(t.relu() @ c.relu().transpose(-2, -1), dim=-1) / scale
    guidance = s + attention @ s
    agka = torch.linalg.vector_norm((guidance - t).flatten(1), dim=1).mean()
    discrepancy = torch.linalg.vector_norm((c - s).flatten(1), dim=1)
    cross_supervision = ((1 - similarity.sigmoid()) * discrepancy).mean()
    return agka, cross_supervision


class CSAKDModel(DistillationModel):
    @staticmethod
    def get_distill_layers(network):
        return module_indices(network) + [detect_head(network).i]

    def _register_feature_hooks(self):
        super()._register_feature_hooks()
        for network, cache, handles in (
            (self.student_model, self._student_feats, self._student_hooks),
            (self.teacher_model, self._teacher_feats, self._teacher_hooks),
        ):
            if network is None:
                continue
            if self.feats_idx[:-1] != module_indices(network):
                raise ValueError("Index modul teacher/student CSAKD berbeda.")
            for index in self.feats_idx[:-1]:
                module = network.model[index]
                # Hapus pre-hook lama jika model pernah di-deserialize.
                for key, hook in list(module._forward_pre_hooks.items()):
                    if isinstance(hook, InputHook):
                        del module._forward_pre_hooks[key]
                handles.append(module.register_forward_pre_hook(InputHook(cache, ("input", index))))

    def __init__(self, teacher_model, student_model):
        super().__init__(teacher_model, student_model)
        inputs, outputs = [], []
        for index in self.feats_idx[:-1]:
            sf, tf = self._student_feats[("input", index)], self._teacher_feats[("input", index)]
            so, to = self._student_feats[index], self._teacher_feats[index]
            if sf.shape[-2:] != tf.shape[-2:] or so.shape[-2:] != to.shape[-2:]:
                raise ValueError("Spatial feature teacher/student tidak seukuran.")
            inputs.append(align_channels(sf.shape[1], tf.shape[1]))
            outputs.append(align_channels(so.shape[1], to.shape[1]))
        self.projector = nn.ModuleDict({"input": nn.ModuleList(inputs), "output": nn.ModuleList(outputs)}).to(
            next(student_model.parameters()).device)
        self._teacher_feats.clear()
        self._student_feats.clear()

    def kd_loss(self):
        components = []
        for pos, index in enumerate(self.feats_idx[:-1]):
            student = self.projector["output"][pos](self._student_feats[index].float())
            incoming = self.projector["input"][pos](self._student_feats[("input", index)].float())
            cross = self.teacher_model.model[index](incoming)  # frozen teacher module, gradient to student
            target = self._teacher_feats[index]
            components.append(csakd_losses(student, target, cross))
        return {
            "kd_agka_loss": sum(x[0] for x in components),
            "kd_cross_loss": sum(x[1] for x in components),
        }

    def loss(self, batch, preds=None):
        return combined_loss(self, batch, preds, {"kd_agka_loss": KD_AGKA_WEIGHT, "kd_cross_loss": KD_CROSS_WEIGHT})


class KDTrainer(DetectionTrainer):
    def set_model_attributes(self):
        super().set_model_attributes()
        if self.args.compile or self.args.resume:
            raise ValueError("CSAKD pilot: compile/resume tidak didukung.")
        self.model = CSAKDModel(self.args.distill_model, self.model)

    def train(self):
        try:
            return super().train()
        finally:
            from ultralytics.utils.torch_utils import unwrap_model
            net = unwrap_model(self.model)
            if hasattr(net, "_remove_feature_hooks"):
                net._remove_feature_hooks()

KD_TRAINER = KDTrainer
KD_TRAIN_ARGS = {"distill_model": str(teacher_path), "dis": 1.0, "compile": False}
KD_CONFIG = {"agka_weight": KD_AGKA_WEIGHT, "cross_weight": KD_CROSS_WEIGHT}

## 6. Model Training

In [ ]:
from datetime import datetime

RUN_NAME = f"{EXPERIMENT_ID}_seed{SEED}_{datetime.now():%Y%m%d_%H%M%S}"
train_args = dict(
    data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    optimizer=OPTIMIZER, lr0=LR, seed=SEED, device=DEVICE,
    weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
    project=PROJECT, name=RUN_NAME, exist_ok=False,
    workers=WORKERS, patience=PATIENCE, plots=True,
    amp=AMP, pretrained=True, save=True,
)
train_args.update(KD_TRAIN_ARGS)

# W&B: satu run dan satu koleksi artifact checkpoint berversi per eksperimen.
wb_run = None
wb_state = {"uploaded_epochs": set(), "logged_epochs": set(), "best_epoch": None}
if WANDB_ENABLED:
    wb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=RUN_NAME,
        job_type="training",
        group="coco8-10epoch-demo" if DATA == "coco8.yaml" else "research",
        config={
            "experiment": EXPERIMENT_ID, "method": METHOD, "model": MODEL_WEIGHTS,
            "teacher_checkpoint": str(TEACHER_CHECKPOINT) if TEACHER_CHECKPOINT else None,
            "dataset": DATA, "eval_split": EVAL_SPLIT, "epochs": EPOCHS,
            "imgsz": IMGSZ, "batch": BATCH, "seed": SEED, "device": str(DEVICE),
            "optimizer": OPTIMIZER, "learning_rate": LR, "amp": AMP,
            "workers": WORKERS, "patience": PATIENCE,
            "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
            "close_mosaic": CLOSE_MOSAIC,
            "kd": KD_CONFIG, "diagnostic_only": DATA == "coco8.yaml",
        },
        tags=[METHOD, EXPERIMENT_ID, "diagnostic" if DATA == "coco8.yaml" else "research"],
    )

    def save_epoch_artifact(trainer):
        """Snapshot checkpoint setiap epoch; upload selesai sebelum last.pt ditimpa epoch berikutnya."""
        epoch = int(trainer.epoch) + 1
        last_path = Path(trainer.last)
        best_path = Path(trainer.best)
        if not last_path.is_file():
            raise RuntimeError(f"Checkpoint epoch {epoch} tidak ada: {last_path}")
        fitness = None if trainer.fitness is None else float(trainer.fitness)
        best_fitness = None if trainer.best_fitness is None else float(trainer.best_fitness)
        is_best = fitness is not None and best_fitness is not None and fitness == best_fitness
        if is_best:
            wb_state["best_epoch"] = epoch
        artifact = wandb.Artifact(
            name=f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints",
            type="model",
            description="Checkpoint penuh per epoch: last.pt, best-to-date.pt, results.csv, args.yaml",
            metadata={
                "epoch": epoch, "fitness": fitness, "best_fitness": best_fitness,
                "is_best_epoch": bool(is_best), "best_epoch_so_far": wb_state["best_epoch"],
                "experiment": EXPERIMENT_ID, "dataset": DATA, "seed": SEED,
            },
        )
        artifact.add_file(str(last_path), name="last.pt")
        if best_path.is_file():
            artifact.add_file(str(best_path), name="best.pt")
        for filename in ("results.csv", "args.yaml"):
            file_path = Path(trainer.save_dir) / filename
            if file_path.is_file():
                artifact.add_file(str(file_path), name=filename)
        aliases = [f"epoch-{epoch:03d}", "latest"]
        if is_best:
            aliases.append("best")
        # Synchronous wait menjamin isi artifact epoch ini selesai dikirim sebelum file lokal berubah.
        wb_run.log_artifact(artifact, aliases=aliases).wait()
        wb_state["uploaded_epochs"].add(epoch)
        print(f"W&B checkpoint: epoch={epoch}, best={is_best}, aliases={aliases}")

    def log_epoch_metrics(trainer):
        """Satu record grafik per epoch (training loss, KD loss, validation, LR, best epoch)."""
        epoch = int(trainer.epoch) + 1
        # Ultralytics memanggil on_fit_epoch_end sekali lagi pada final_eval; abaikan panggilan ekstra.
        if epoch not in wb_state["uploaded_epochs"] or epoch in wb_state["logged_epochs"]:
            return
        loss = trainer.label_loss_items(trainer.tloss, prefix="train")
        payload = {"epoch": epoch, **{k: float(v) for k, v in loss.items()},
                   **{k: float(v) for k, v in trainer.lr.items()}}
        for key, value in trainer.metrics.items():
            try:
                payload[key] = float(value)
            except (ValueError, TypeError):
                continue
        payload["checkpoint/is_best_epoch"] = int(wb_state["best_epoch"] == epoch)
        payload["checkpoint/best_epoch_so_far"] = wb_state["best_epoch"] or 0
        wb_run.log(payload, step=epoch)
        wb_state["logged_epochs"].add(epoch)

    model.add_callback("on_model_save", save_epoch_artifact)
    model.add_callback("on_fit_epoch_end", log_epoch_metrics)

print("Training:", EXPERIMENT_ID, "| data:", DATA, "| epochs:", EPOCHS,
      "| W&B:", "online + every epoch artifact" if WANDB_ENABLED else "disabled")
start_time = time.perf_counter()
try:
    if KD_TRAINER is None:
        model.train(**train_args)
    else:
        model.train(trainer=KD_TRAINER, **train_args)
except BaseException:
    if wb_run is not None:
        wb_run.finish(exit_code=1)
    raise
training_seconds = time.perf_counter() - start_time
run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / "weights" / "best.pt"
if not best_pt.is_file():
    raise FileNotFoundError(f"Checkpoint best.pt tidak ditemukan: {best_pt}")
if wb_run is not None and len(wb_state["uploaded_epochs"]) != EPOCHS:
    print("Catatan: jumlah epoch selesai berbeda dari target (misalnya early stopping).",
          len(wb_state["uploaded_epochs"]), "/", EPOCHS)
print("Training selesai:", run_dir)


## 7. Model Evaluation

In [ ]:
# Evaluasi dari checkpoint terbaik (bukan last.pt).
best_model = YOLO(str(best_pt))
metrics = best_model.val(
    data=DATA, split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH,
    device=DEVICE, plots=True, verbose=False,
)

from ultralytics.utils.torch_utils import get_flops
parameters = sum(p.numel() for p in best_model.model.parameters())
try:
    gflops = float(get_flops(best_model.model, imgsz=IMGSZ))
except (AttributeError, ImportError, RuntimeError, TypeError, ValueError) as error:
    print("GFLOPs tidak tersedia:", error)
    gflops = None

results = {
    "experiment": EXPERIMENT_ID,
    "method": METHOD,
    "model": MODEL_WEIGHTS,
    "dataset": DATA,
    "eval_split": EVAL_SPLIT,
    "seed": SEED,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "optimizer": OPTIMIZER,
    "learning_rate": LR,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
    "parameters": int(parameters),
    "gflops": gflops,
    "inference_ms_per_image": float(metrics.speed.get("inference", 0.0)),
    "training_seconds": float(training_seconds),
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "teacher_checkpoint": str(TEACHER_CHECKPOINT) if METHOD != "none" else None,
    "best_pt": str(best_pt),
}
if torch.cuda.is_available():
    results["peak_gpu_memory_gb"] = float(torch.cuda.max_memory_allocated() / 1024**3)
print(json.dumps(results, indent=2, ensure_ascii=False))

## 8. Results & Outputs

In [ ]:
# File lokal tetap tersimpan oleh Ultralytics, W&B menyimpan versi artifact tiap epoch.
run_dir.mkdir(parents=True, exist_ok=True)
results["best_epoch"] = wb_state["best_epoch"] if WANDB_ENABLED else None
(run_dir / "metrics.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
configuration = {
    "experiment": EXPERIMENT_ID, "dataset": DATA, "evaluation_split": EVAL_SPLIT,
    "model": MODEL_WEIGHTS, "teacher": TEACHER_CHECKPOINT if METHOD != "none" else None,
    "seed": SEED, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
    "optimizer": OPTIMIZER, "lr0": LR, "device": DEVICE,
    "workers": WORKERS, "patience": PATIENCE, "amp": AMP,
    "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
    "close_mosaic": CLOSE_MOSAIC,
    "kd_hyperparameters": KD_CONFIG,
    "wandb_enabled": WANDB_ENABLED, "wandb_project": WANDB_PROJECT,
}
(run_dir / "config.json").write_text(json.dumps(configuration, indent=2, ensure_ascii=False), encoding="utf-8")

if wb_run is not None:
    summary = {f"final/{key}": val for key, val in results.items()
               if isinstance(val, (int, float)) and not isinstance(val, bool) and val is not None}
    wb_run.log(summary)
    wb_run.summary.update(summary)
    wb_run.summary["best_epoch"] = wb_state["best_epoch"]
    wb_run.summary["checkpoint_collection"] = f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints"
    wb_run.summary["uploaded_checkpoint_epochs"] = len(wb_state["uploaded_epochs"])

    # Ringkasan/config dan grafik sebagai Artifact terpisah, tanpa menggandakan checkpoint model.
    final_artifact = wandb.Artifact(
        name=f"{EXPERIMENT_ID}-{wb_run.id}-results", type="results",
        metadata={"experiment": EXPERIMENT_ID, "best_epoch": wb_state["best_epoch"]},
    )
    for filename in ("metrics.json", "config.json", "results.csv", "args.yaml"):
        candidate = run_dir / filename
        if candidate.is_file():
            final_artifact.add_file(str(candidate), name=filename)
    for folder_name, folder in (("training", run_dir),
                                ("evaluation", Path(getattr(metrics, "save_dir", run_dir)))):
        if folder.exists():
            for plot_path in sorted(folder.glob("*.png")):
                key = f"plots/{folder_name}/{plot_path.stem}"
                wb_run.log({key: wandb.Image(str(plot_path))})
                final_artifact.add_file(str(plot_path), name=f"{folder_name}/{plot_path.name}")
    wb_run.log_artifact(final_artifact, aliases=["final"]).wait()
    print("W&B:", wb_run.url)
    print("Checkpoint alias: epoch-001 ... epoch-%03d, latest, best" % len(wb_state["uploaded_epochs"]))
    wb_run.finish()

print("Hasil lokal:", run_dir)
print("  - weights/best.pt, weights/last.pt")
print("  - results.csv, args.yaml")
print("  - metrics.json, config.json")
